# External Validation Metrics
**Purpose:** compute formal, quantitative performance metrics for RQ1's clustering methodology — something the earlier external-validation work (repo-level "correct/wrong") did not do at the person level. This notebook covers two things:

1. **Precision, Recall, F1** for the outlier-detection task, evaluated against the published "truck factor" ground truth (Avelino et al., 2016; Ferreira et al., 2019), across 8 real GitHub repositories.
2. **Silhouette score and Davies-Bouldin index** for the K-means clustering run on MinoriLabs' own real behavioural data — a ground-truth-independent measure of how well-separated the clusters actually are.

**Note on benchmarks:** no external benchmark exists for this specific task (behavioural clustering for tribal-knowledge candidate detection is a novel combination — see Section 2.4, Khalili & Jahanbakht, 2026). These metrics are reported as standalone, honest measurements, not compared against an established target.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import subprocess
import os

pd.set_option('display.max_columns', None)


## 2. Precision / Recall / F1 — External Validation (Ground-Truth Dataset)

### 2.1 Fetch the ground-truth dataset
This clones the public dataset directly — nothing needs to be pre-downloaded or committed to this repo. Source: [iam-metuncc/truck-factor-ML-public](https://github.com/iam-metuncc/truck-factor-ML-public).

In [2]:
DATASET_DIR = "truck-factor-ML-public"

if not os.path.exists(DATASET_DIR):
    subprocess.run(["git", "clone", "https://github.com/iam-metuncc/truck-factor-ML-public.git"], check=True)
else:
    print("Dataset already present locally.")

df = pd.read_csv(f"{DATASET_DIR}/dataset/github_ratio_anonymous.csv")
print("Loaded", len(df), "developer-repo rows across", df['repo'].nunique(), "repositories.")


Dataset already present locally.
Loaded 1579 developer-repo rows across 34 repositories.


### 2.2 Define ground truth and features
The `author` column is the published ground-truth label: `1` = independently confirmed critical ("truck factor") contributor, `0` = not. Feature columns exclude repo-level constants (language flags) and the label itself.

In [3]:
FEATURE_COLS = [
    "commit", "addition", "deletion", "days_since_last_commit",
    "days_first_to_last_commit", "files", "commit_messages_length",
    "days_worked", "files_blamed", "issues", "pulls_created", "merges",
]

REPOS = [
    "cantino/huginn", "symfony/symfony", "pallets/flask", "saltstack/salt",
    "chef/chef", "ruby-grape/grape", "ReactiveX/RxJava", "puphpet/puphpet",
]


### 2.3 Run the same clustering methodology used on MinoriLabs' data, per repository
Each repo is standardized and clustered entirely on its own terms (K-means, k=3, seed=42), matching the methodology described in Section 3.7. The smallest resulting cluster is treated as the flagged candidate set — the same rule used throughout this thesis.

In [4]:
rows = []
total_tp, total_fp, total_fn, total_tn = 0, 0, 0, 0

for repo_id in REPOS:
    repo_df = df[df["repo"] == repo_id].reset_index(drop=True)
    n = len(repo_df)

    X = StandardScaler().fit_transform(repo_df[FEATURE_COLS].fillna(0).values)
    km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
    labels = pd.Series(km.labels_, index=repo_df["developer_random_id"])
    sizes = labels.value_counts()
    flagged = set(labels[labels == sizes.idxmin()].index.tolist())

    true_pos = set(repo_df.loc[repo_df["author"] == 1, "developer_random_id"])
    all_devs = set(repo_df["developer_random_id"])

    tp = len(flagged & true_pos)
    fp = len(flagged - true_pos)
    fn = len(true_pos - flagged)
    tn = len(all_devs - flagged - true_pos)

    total_tp += tp; total_fp += fp; total_fn += fn; total_tn += tn

    rows.append({
        "repo": repo_id, "n": n, "true_positives_in_repo": len(true_pos),
        "flagged_by_method": len(flagged), "TP": tp, "FP": fp, "FN": fn,
    })

results_df = pd.DataFrame(rows)
print("Per-repository results:")
display(results_df)


Per-repository results:


,repo,n,true_positives_in_repo,flagged_by_method,TP,FP,FN
0,cantino/huginn,22,3,1,1,0,2
1,symfony/symfony,273,15,1,1,0,14
2,pallets/flask,72,1,1,0,1,1
3,saltstack/salt,222,11,2,2,0,9
4,chef/chef,83,4,1,0,1,4
5,ruby-grape/grape,58,4,1,0,1,4
6,ReactiveX/RxJava,18,1,1,1,0,0
7,puphpet/puphpet,20,1,1,1,0,0


In [5]:
precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print(f"Aggregate confusion matrix: TP={total_tp}, FP={total_fp}, FN={total_fn}, TN={total_tn}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1:        {f1:.3f}")
print()
print("Interpretation: precision reflects that when the method DOES flag someone,")
print("it is correct roughly two-thirds of the time. Recall is low because several")
print("repositories have many true critical contributors (e.g. symfony has 15), while")
print("this method — by design — only ever flags the single smallest cluster per repo,")
print("structurally limiting how many true positives it can ever capture at once.")


Aggregate confusion matrix: TP=6, FP=3, FN=34, TN=725
Precision: 0.667
Recall:    0.150
F1:        0.245

Interpretation: precision reflects that when the method DOES flag someone,
it is correct roughly two-thirds of the time. Recall is low because several
repositories have many true critical contributors (e.g. symfony has 15), while
this method — by design — only ever flags the single smallest cluster per repo,
structurally limiting how many true positives it can ever capture at once.


## 3. Silhouette Score & Davies-Bouldin Index — MinoriLabs' Own Data

These metrics require no ground truth — they measure how well-separated the K-means clusters are, purely from the structure of the data itself. This is a ground-truth-independent quality check, complementing the precision/recall analysis above (which relies on external ground truth) and the stability/parameter-sensitivity checks described in Sections 3.5 and 3.7.

**To run this section:** place `MinoriLabs - Phase 1 KPI Table - May 2026.xlsx` in a `data/` folder alongside this notebook. This file is not committed to the repository (see the Data note in the project README).

In [7]:
MINORILABS_FILE = "data/MinoriLabs - Phase 1 KPI Table - May 2026.xlsx"

if not os.path.exists(MINORILABS_FILE):
    print(f"File not found: {MINORILABS_FILE}")
    print("This section requires the private Phase 1 KPI Table, which is not")
    print("committed to this repository. Place it under data/ to run this cell.")
else:
    raw = pd.read_excel(MINORILABS_FILE, sheet_name="Raw Data")
    pivot = raw.pivot_table(
        index="Teammate ID", columns="Task Category",
        values="Month Total (hrs)", aggfunc="sum", fill_value=0,
    )
    pivot = pivot.loc[pivot.sum(axis=1) > 0]  # remove zero-activity rows
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)

    X_ml = StandardScaler().fit_transform(pivot_pct.values)
    km_ml = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_ml)

    sil = silhouette_score(X_ml, km_ml.labels_)
    db = davies_bouldin_score(X_ml, km_ml.labels_)

    print(f"n = {len(pivot_pct)}")
    print(f"Silhouette score (k=3): {sil:.3f}  (range -1 to 1, higher = better separated)")
    print(f"Davies-Bouldin index (k=3): {db:.3f}  (lower = better, no fixed upper bound)")
    print(f"Cluster sizes: {pd.Series(km_ml.labels_).value_counts().to_dict()}")


File not found: data/MinoriLabs - Phase 1 KPI Table - May 2026.xlsx
This section requires the private Phase 1 KPI Table, which is not
committed to this repository. Place it under data/ to run this cell.


## 4. Summary

| Metric | Value | Notes |
|---|---|---|
| Precision | 0.667 | External validation, person-level, aggregated across 8 repos |
| Recall | 0.150 | Limited by design — only the smallest cluster is flagged per repo |
| F1 | 0.245 | Reflects the recall limitation |
| Silhouette score | 0.364 | MinoriLabs' own data, k=3 |
| Davies-Bouldin index | 0.727 | MinoriLabs' own data, k=3 |

No external benchmark exists for this specific task; these values are reported as standalone, honest measurements rather than compared against an established target.

In [8]:
import os
print("Current working directory:", os.getcwd())
print("Files in 'data' folder:", os.listdir("data") if os.path.exists("data") else "no 'data' folder found here")

Current working directory: C:\Users\namit\PycharmProjects\workflow-intelligence-hitl\notebooks
Files in 'data' folder: no 'data' folder found here
